# Training Series: Introduction to Discontinuous Galerkin Methods for Flow Problems

Goals of this training series:
- Basic understanding of Discontinuous Galerkin methods.
- Familiarize with the application of DG methods to flow problems (incompressible flows)
- Developing an own incompressible single-phase flow solver
- Introduction to the BoSSS-framework and Jupyter-notebooks

## 1 Approximation of functions by broken polynomials

### 1.1 Definition of a grid in one dimension

- Starting point: a domain $\Omega$ in which we want to approximate a function $f(x)$, or solve a partial differential equation (PDE). 

- Grid as a set of cells $K_0, ..., K_{J-1}$

- 1D-example: an interval $\Omega = (x_{\textrm{left}}, x_{\textrm{right}})$, all cells have the same size: 

    <img src="images/1DgridExample.PNG" style="width: 600px;"/>

    $h$ is the characteristic length-scale of the grid

- Cells are open sets: $K_j = (a_j, a_{j+1}) = \{ x; a_j < x < a_{j+1}\}$

- The corresponding closure: $\bar{K}_j = [ a_j, a_{j+1} ] = \{ x; a_j \leq x \leq a_{j+1}\}$

- The domain $\Omega$ is also an open set, as usual in the theory of PDEs.

- Common properties of grids (also 2D, 3D):
    - Cells do not overlap: $K_1 \cap K_2 = \emptyset$ resp. $\int_{\bar{K}_1 \cap \bar{K}_2} 1 \textrm{d}V = 0$
    
    - Cells cover the entire domain: $\bar{\Omega} = \cup_{j=0}^{J-1} \bar{K_j}$

### 1.2 Polynomial basis

- A polynomial basis for the polynomial degree $k$ is a set of polynomial functions $\Phi_0(\vec{x}), ..., \Phi_{N_k-1}(\vec{x})$, which are linearly independent.

- Two function $f(x)$ and $g(x)$ are linear independent, if (and only if) there is no constant α so that $f(x) = αg(x)$ for all $x$ in the domain.

- $N_k$ is the number of monomials up to degree $k$ depending on the spatial dimension $D$:

    | $k$ | $D=1$ | $D=2$ | $D=3$ |
    |:---:|:-----:|:-----:|:-----:|
    |  0  |   1   |   1   |   1   |
    |  1  |   2   |   3   |   4   |
    |  2  |   3   |   6   |   10  |
    |  3  |   4   |   10  |   20  |
    |  4  |   5   |   15  |   35  |
    |  :  |       |       |       |
    | $k$ | $k+1$ | $\frac{(k+1)^2+k+1}{2}$ | $\frac{1}{6}k^3 + k^2 \frac{11}{6}k + 1$ |    

Now, we want to approximate some function $u(\vec{x})$ by a polynomial $u_h(\vec{x})$, i.e. $u(\vec{x}) ≈ u_h(\vec{x})$, using our polynomial
basis

>$$ \vec{\Phi} = (\Phi_0, ..., \Phi_{N_k-1}). $$

We can represent $u_h(\vec{x})$ as a linear combination:

>$$ u_h(\vec{x}) = \sum_{n=0}^{N_k-1} \Phi_n(\vec{x}) \tilde{u}_n = \vec{\Phi} \cdot \vec{\tilde u}. $$

We need this Ansatz to work with $u_h(\vec{x})$ in a computer:
- We implement a code for the basis functions $\Phi_n(\vec{x})$, e.g. $\Phi_2(x) = \left(x^2 − \frac{1}{3}\right) \frac{3\sqrt{10}}{4}$. The basis functions are polynomials themselves, and they are fixed in the computer program.

- During the simulation, we (resp. the computer) only change the numbers $\tilde{u}_n$.

**Examples for a polynomial basis in 1D:**

- Monomials up to degree $k$, i.e. $\Phi_n = x^n$, $n<k$ with $N_k = k+1$ (in 1D)

- *Legendre-polynomials*, also called modal polynomials (solution of Legdendre ODE)

>$$ P_0(x) = 1 $$
>$$ P_1(x) = x $$

>$$ P_{n+1}(x) = \frac{1}{(n+1)}((2n+1) x P_n(x) n P_{n-1}) \quad \textrm{ for } n \geq 1$$
<img src="images/LegendrePolynomials.PNG" style="width: 600px;"/>

Distinct property: All functions are *orthogonal* (will be defined more precise later)

>$$ \int_{x=-1}^{1} P_n(\vec{x})P_m(\vec{x}) = 0 \quad n,m \textrm{ with } n \neq m $$

- *Lagrange-polynomials*, also *nodal polynomials*, ...

**Remarks:** The choice of polynomials

- has no impact on abstract numerical properties like convergence or stability (to be defined)

- is important for round-off errors; in exact architectures all basis choices would behave the same way

- is important for implementation efficiency: especially in 2D and 3D, some representations can be implemented more efficiently than others.

- all different choices for a basis, for the same degree, have the same number of elements; this is called the dimension of the basis.

### 1.3 Approximation of some function $f(x)$ by error minimization (projection)

**General approach**

- We want to find the best approximation $f_h \in \mathbb{P}_k(\{\Omega\})$ to $f \in L^2(\Omega)$ in the following sense:

>$$ \int_{\Omega} {\underbrace{(f_h(x) - f(x))}_{=: r(x)}}^2 \textrm{d}V = \| f_h - f \|_2^2 \rightarrow \textrm{min} $$

- Minimization is equivalent to requiring 

>$$ \langle  r(x) | \Phi_m \rangle = \langle f_h(x) - f(x) | \Phi_m \rangle \stackrel{!}{=} 0 \quad \forall \Phi_m$$

- Verbally: Find $f_h(x)$ such that the residual $r(x)$ is orthogonal to our Ansatz space w.r.t. the scalar product $\langle \cdot | \cdot \rangle$

    <img src="images/Projection.PNG" style="width: 600px;"/>

- Our Ansatz $f_h(x) = \sum_{n=0}^{N_k - 1} \Phi_n(x) \tilde f_n$ leads to the linear system 

>$$ \underbrace{\begin{bmatrix} \langle \Phi_0 | \Phi_0 \rangle & \cdots & \langle \Phi_0 | \Phi_{N_k-1} \rangle \\ \vdots & \ddots & \vdots \\ \langle \Phi_{N_k-1} | \Phi_0 \rangle & \cdots & \langle \Phi_{N_k-1} | \Phi_{N_k-1} \rangle \end{bmatrix}}_{=:\langle \vec{\Phi}^{\textrm{T}} | \vec{\Phi} \rangle} \cdot \underbrace{\begin{pmatrix} \tilde f_0 \\ \vdots \\ \tilde f_{N_k-1} \end{pmatrix}}_{=: \tilde f} = \underbrace{\begin{pmatrix} \langle f | \Phi_0 \rangle  \\ \vdots \\ \langle f | \Phi_{N_k-1} \rangle \end{pmatrix}}_{=: \langle f | \vec{\Phi}^{\textrm{T}} \rangle}  $$

i.e we obtain the coordinate vector $\tilde f$ of $f_h$ as 

>$$ \tilde f = \left( \langle \vec{\Phi}^{\textrm{T}} | \vec{\Phi} \rangle \right)^{-1} \langle f | \vec{\Phi}^{\textrm{T}} \rangle $$

- This defines the projection operator 

>$$ L^2(\Omega) \ni f \mapsto \pi_k(f) = f_h \in \mathbb{P}_k(\mathfrak{K}_h) $$

which is given uniquely by the property

>$$ \langle f_h - f | g_h \rangle = 0 \quad \forall g_h \in \mathbb{P}_k(\mathfrak{K}_h)$$

and can explicitly be computed for a given basis $\vec{\Phi}$

>$$ f \mapsto \pi_k(f) := \vec{\Phi} \cdot \underbrace{\left( \left( \langle \vec{\Phi}^{\textrm{T}} | \vec{\Phi} \rangle \right)^{-1} \langle f | \vec{\Phi}^{\textrm{T}} \rangle \right)}_{= \tilde f} = f_h $$

- The last two equations define the $L^2$*-orthogonal projection* of $f$ onto the approximation space $\mathbb{P}_k(\Omega)$

- General definition of a projector operator: A mapping $\pi$ is called a projector if, and only if it is linear
and idempotent ($\pi^2 = \pi$). (Applying the operator twice is the same as applying it once.)

**Approximation error of the projection**

- Application of the Bramble-Hilbert lemma: Let $f$ be $k$ times continuously differentiable. Then

>$$ \| f(x) - \pi_k(f(c)) \|_{L^2(\Omega)} \leq C \cdot h^{k+1} $$

with a constant $C$ that depends on $f$ but not on $h$.

- Verbally: If $f$ is sufficiently smooth, the approximation error is of order $O(h^{k+1})$, which is one of the major motivations for using higher order methods

- The differentiability assumption is essential. If $f$ is non-smooth, the so-called *Gibbs phenomenon* occurs, which is one of the major drawbacks of higher order methods

- Example: Approximation of the Heaviside function

>$$ H(x) = \begin{cases} 0 \quad x < 0 \\ 1 \quad x > 0\end{cases} $$

by polynomials of various degrees:

<img src="images/HeavisideApproximation.PNG" style="width: 1200px;"/>

Note that the $L^2$ error does decrease when increasing $k$, even though extremely slowly

- The magnitude of the modes $\tilde f_n$ decays rapidly, but only if $f$ is sufficiently smooth (cf. Hands-on 1). This can also be used as a smoothness detector for problems involving shocks or other discontinuities.

### 1.4 From one cell to many cells, extending to 2D and 3D

- 1D: So far we only considered a single cell, e.g. $K = (-1,1)$.

- 2D, 3D: Common reference cell types: triangles, quadrilaterals, tetrahedrons, hexahedrons, prisms

- 1D: We have a mesh like in section 1.1, e.g. cell $K_j = (a_j, a_{j+1}), j = 0, . . . , J − 1$.

- 2D, 3D: Consider a grid $\mathfrak{K}_h$ consisting of $D$-dimensional grid cells $K_j, j = 0, . . . , J − 1$

- 1D: Size of cell $K_j$ is $a_{j+1} − a_j$.

- Different choices for the characteristic mesh size $h$ exist. Common choices are the inner/outer cell diameter,
or the length of longest/shortest edge of the cell

- 1D: Introduce reference cell $K^{\textrm{ref}} = (−1, 1)$ and mapping

>$$ T_j(\xi) = \frac{a_{j+1} - a_j}{2}\xi + \frac{a_{j+1} + a_j}{2} $$

which maps the reference cell $K^{\textrm{ref}}$ to the physical cell $K_j$.

<img src="images/1DreferenceElement.PNG" style="width: 600px;"/>

- 2D, 3D: Analogous; our reference cell may be a unit square, triangle, etc. we have a linear mapping

>$$ T_j(\vec{\xi}) = \mathbf{A}_j \vec{\xi} + \vec{o}_j $$

which maps the reference cell to physical cells.

<img src="images/2DreferenceElement.PNG" style="width: 600px;"/>

- Note: coordinates on reference cell $\xi$ (in 1D), resp. $\vec{\xi} = (\xi, \eta)^{\textrm{T}}$, $\vec{\xi} = (\xi, \eta, \nu)^{\textrm{T}}$ (in 2D, 3D); coordinates in physical cell $x$, resp. $\vec{x} = (x, y)^{\textrm{T}}$ resp. $\vec{x} = (x, y, z)^{\textrm{T}}$ .

- In a first step, we define a DG basis on the reference element, e.g. Legendre polynomials $\Phi_n(\xi) = P_n(\xi)$.

- In BoSSS, specifically we use the following orthonormal polynomials, in 1D

>$$ \Phi_0 = \sqrt{2}/2 $$
>$$ \Phi_1 = \xi\sqrt{6}/2 $$
>$$ \Phi_2 = (3/4)(\xi^2 - 1/3)\sqrt{10} $$
>$$ \Phi_3 = (5/4)(\xi^3 - (3/5)\xi)\sqrt{14} $$
>$$ \Phi_4 = ... $$

and in 2D (on the square $K^{\textrm{ref}} = (−1, 1)^2$) we use

>$$ \Phi_0 = 1/2 $$
>$$ \Phi_1 = (1/2)\xi\sqrt{3} $$
>$$ \Phi_2 = (1/2)\eta\sqrt{3} $$
>$$ \Phi_3 = (1/4)(3\xi^2 - 1)\sqrt{5} $$
>$$ \Phi_4 = (3/2)\xi \eta $$
>$$ \Phi_5 = (1/4)(3\eta^2 - 1)\sqrt{5} $$
>$$ \Phi_6 = ... $$

These polynomials can be generated automatically by the Gram-Schmidt algorithm, using a Computer-
Algebra System.

- **TODO** Orthonormality property

- Then, we define the basis on the physical cell $K_j$, on the entire grid:

>$$ \Phi_{j,n}(x) = \begin{cases} \alpha_{j,n} \Phi_n(T_j^{-1}(x)) \quad \textrm{for } x \in K_j \\ 0 \quad \textrm{otherwise} \end{cases} $$

The scaling factor $\alpha_{j,n}$ may be set so that **(TODO: Update!)**

>$$ \| \Phi_{j,n} \|_{L^2(\Omega)} = \left( \int_{a_0}^{a_J} \Phi_{j,n}^2 \textrm{d}x \right)^{1/2} = \left( \int_{a_0}^{a_J} \Phi_{j,n}^2 \textrm{d}V  \right)^{1/2} = 1 $$

<img src="images/ReferenceElement.PNG" style="width: 400px;"/>

<img src="images/MeshElements.PNG" style="width: 800px;"/>

- Using this, we can write the DG approximation as

>$$ f_h(x) = \sum_{j=0}^{J-1} \sum_{n=0}^{N_k-1} \Phi_{j,n}(x) \tilde f_{j,n} $$

- Note: for $l \neq j$, we obviously have orthogonality, i.e.

>$$ \langle \Phi_{j,n} | \Phi_{l,m} \rangle_{\Omega} = 0 $$


### Hands-on 1

## 2 DG for first order problems

- Follow the same idea (finding the best approximation $f_h \in \mathbb{P}_k(\mathfrak{K}_h)$ of a function $f \in L^2(\Omega)$ by error minimization) for some conservation law

>$$ \partial_t c + \nabla \cdot \vec{f}(c) = 0 $$

with suitable initial and boundary conditions for the residual

>$$ r(c_h) = \partial_t c_h + \nabla \cdot \vec{f}(c_h) = 0 $$

for some approximation $c_h$ of $c$. That is, find the best approximation $c_h \in \mathbb{P}_k(\mathfrak{K}_h)$ such that

>$$ \langle r(c_h) | \Phi_{j,m} \rangle_{K_j} = 0 \quad \forall \Phi_{j,m}$$

in the given cell $K$

### 2.1 The edges of the computational grid

<img src="images/ComputationalGrid.PNG" style="width: 600px;"/>

- Extending the definitions in section 1.1 to multiple dimensions, we define a computational grid $\mathfrak{K}_h := {K_0, K_1, . . . , K_{J−1}}$ as a set of $D$-dimensional mesh cells $K_j$ (usually polytopes) that cover the computational domain $\Omega$

- For each cell $K_j$, we define the associated normal vectors $\vec{n}_{\partial K_j}$ on $\partial K_j$ such that they are pointing outwards. As a result, the direction of a normal vector is not unique on some edge, but depends on the particular cell under consideration.

- Note: The above definitions are sensible for deriving the local form of a DG scheme, which we will outline in the following subsections. This local form is extremely helpful for understanding the general idea of DG methods. However, efficient implementations of DG schemes should follow a different approach. Some of the associated issues will thus be discussed in Section 4

- Each grid cell $K_j$ is associated with its own, *cell-local* basis $\vec{\Phi}_j = (\Phi_{j,n})_{n=0,...,N_{k−1}}$. The locality of the basis stems from the fact that each basis function is only non-zero in its associated cell, i.e.

>$$ \textrm{supp}(\Phi_{j,n}) = \bar{K}_j, $$

where $\textrm{supp}(\Phi_{j,n}) $ denotes the *support* of $\Phi_{j,n}$.

### 2.2 Semi-discrete weak formulation

**Derivation**

- We start from the conservation law 

>$$ \partial_t c + \nabla \cdot \vec{f}(c) = 0 $$

- We require that the residual 

>$$ r(c_h) := \frac{\partial c_h}{\partial t} + \nabla \cdot \vec{f}(c_h) $$

is orthogonal to some *test function* $\Phi_{j,m}. This leads to 

>\begin{align}
\langle r(c_h) | \Phi_{j,m} \rangle_{K_j} &= \langle \partial_t c_h + \nabla \cdot \vec{f}(c_h)  | \Phi_{j,m} \rangle_{K_j} \\
&= \langle \partial_t c_h | \Phi_{j,m} \rangle_{K_j} + \langle \nabla \cdot \vec{f}(c_h)  | \Phi_{j,m} \rangle_{K_j} \\
&\stackrel{!}{=} 0 \quad \forall \Phi_{j,m} \\
\end{align}

in each cell $K_j$

- Note: The index $m$, $0 \leq m \leq N_k - 1$ correspond to the $m$-th *test function* $\Phi_{j,m}$ in cell $K_j$

- Reminder: Inserting the definition of the scalar product, this is equivalent to

>\begin{align}
0 &\stackrel{!}{=} \langle \partial_t c_h + \nabla \cdot \vec{f}(c_h) | \Phi_{j,m} \rangle_{K_j} \\
&= \int_{K_j} \left( \partial_t c_h + \nabla \cdot \vec{f}(c_h) \right) \Phi_{j,m} \textrm{d}V \\
&= \underbrace{\int_{K_j} \left( \partial_t c_h \right) \Phi_{j,m} \textrm{d}V}_{\textrm{Temporal term}} + \underbrace{\int_{K_j} \left( \nabla \cdot \vec{f}(c_h) \right) \Phi_{j,m} \textrm{d}V}_{\textrm{Spatial term}}  \quad \forall \Phi_{j,m}
\end{align}

- For the spatial term, we use integration by parts to obtain

>$$ \int_{K_j} \left( \nabla \cdot \vec{f}(c_h) \right) \Phi_{j,m} \textrm{d}V = \underbrace{\oint_{\partial K_j} \left( \vec{f}(c_h) \cdot \vec{n}_{\partial K_j} \right) \Phi_{j,m} \textrm{d}A}_{\textrm{Surface term}} - \underbrace{\int_{K_j} \vec{f}(c_h)  \cdot \nabla \Phi_{j,m} \textrm{d}V}_{\textrm{Volume term}} \quad \forall \Phi_{j,m}$$

- In sum, we have

>$$ \underbrace{\int_{K_j} \left( \partial_t c_h \right) \Phi_{j,m} \textrm{d}V}_{\textrm{Temporal term}} + \underbrace{\oint_{\partial K_j} \left( \vec{f}(c_h) \cdot \vec{n}_{\partial K_j} \right) \Phi_{j,m} \textrm{d}A}_{\textrm{Surface term}} - \underbrace{\int_{K_j} \vec{f}(c_h)  \cdot \nabla \Phi_{j,m} \textrm{d}V}_{\textrm{Volume term}} = 0\quad \forall \Phi_{j,m} $$

**Galerkin Ansatz**

- Galerkin approach: Use identical trial/Ansatz and test functions by defining the local solution

> $$ c_{h,j}(\vec{x}, t) = \sum_{n=0}^{N_k - 1} \Phi_{j,n}(\vec{x}) \tilde c_{j,n}(t) = \vec{\Phi}_j(\vec{x}) \cdot \tilde c_j(t)

(which also defines the global solution according).

- The yet unknown coefficients $\tilde c_{j,n}(t)$ are called the degrees of freedom (DOF)

- For each cell, we have $N_k$ DOF (one for each coefficient, respectively trial function) and $N_k$ equations (one for each test function). The total number of DOF/equations in the system is thus $J \cdot N_k$, where $J$ is the total number of cells

**Temporal term**

- Applying the *Galerkin Ansatz* into the temporal term leads to 

>\begin{align} 
\int_{K_j} \partial_t c_h \Phi_{j,m} \textrm{d}V &\approx \int_{K_j} \partial_t c_{h,j} \Phi_{j,m} \textrm{d}V \\
&= \int_{K_j} \partial_t \left( \sum_{n=0}^{N_k - 1} \Phi_{j,n} \tilde c_{j,n} \right) \Phi_{j,m} \textrm{d}V \\
&= \int_{K_j} \sum_{n=0}^{N_k - 1} ( \partial_t \tilde c_{j,n} ) \Phi_{j,n} \Phi_{j,m} \textrm{d}V \\
&= \sum_{n=0}^{N_k - 1} \partial_t \tilde c_{j,n} \underbrace{\int_{K_j}  \Phi_{j,n} \Phi_{j,m} \textrm{d}V}_{=: \mathbf{M}_{(j,n)(j,m)}} \\
\end{align}

- The *mass matrix* of cell $K_j$, 

>$$ \mathbf{M}_j := \mathbf{M}_{(j,-)(j,-)} $$

only depends on the choice of the *cell-local* basis functions:

>$$ \mathbf{M}_j = \begin{bmatrix} 
\langle \Phi_{j,0} | \Phi_{j,0} \rangle_{K_j} & \langle \Phi_{j,0} | \Phi_{j,1} \rangle_{K_j} & \cdots \\
\langle \Phi_{j,1} | \Phi_{j,0} \rangle_{K_j} & \langle \Phi_{j,1} | \Phi_{j,1} \rangle_{K_j} & \cdots \\
\vdots & \vdots & \ddots
\end{bmatrix}$$

in particular, $\mathbf{M}_j $ does not depend on neighboring cells, which implies that the global mass matrix is block-diagonal:

>$$ \mathbf{M}_j = \begin{bmatrix} 
\mathbf{M}_0  & 0 & 0 & \cdots \\
0 & \mathbf{M}_1 & 0 & \cdots \\
0 & 0 & \mathbf{M}_2 & \cdots \\
\vdots & \vdots & \vdots & \ddots
\end{bmatrix}$$

- All in all, we can write

>$$ \int_{K_j} \partial_t c_h \Phi_{j,m} \textrm{d}V = \sum_{n=0}^{N_k - 1} \partial_t \tilde c_{j,n} \mathbf{M}_{(j,n)(j,m)} = \mathbf{M}_j \partial_t \tilde c_j $$

**Surface term**

- The surface term is given by

>$$ \oint_{\partial K_j} (\vec{f}(c_h) \cdot \vec{n}_{\partial K_j}) \Phi_{j,m} \textrm{d}A $$

- Inserting the Ansatz function is problematic since $\partial K_j$ is shared by other cells, but we do not enforce continuity! Thus, there is a difference between the inner value

>$$ c_{h,j}^{-} = c_{h,j}^{-}(\vec{x},t) := \lim_{\epsilon \rightarrow 0^+} c_h(\vec{x} - \epsilon \vec{n}_{\partial K_j}, t)  $$

and the outer value

>$$ c_{h,j}^{+} = c_{h,j}^{+}(\vec{x},t) := \lim_{\epsilon \rightarrow 0^+} c_h(\vec{x} + \epsilon \vec{n}_{\partial K_j}, t)  $$

for all points $\vec{x}$ on all edges that do not coincide with a domain boundary

<img src="images/NumericalFlux.PNG" style="width: 800px;"/>

- Solution: Introduce a *numerical flux* function

>$$ \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \approx \vec{f}(c_h) \cdot \vec{n}_{\partial K_j} $$

that defines a unique value on $\partial K_j$

- The numerical flux couples the DOF of neighboring cells. It should satisfy certain mathematical and
physical properties, which will be discussed in detail in Section 3.

- In sum, we have

>$$ \oint_{\partial K_j} (\vec{f}(c_h) \cdot \vec{n}_{\partial K_j}) \Phi_{j,m} \textrm{d}A \approx \oint_{\partial K_j} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \Phi_{j,m} \textrm{d}A $$

**Volume term**

**Putting everything together**

- The semi-discrete weak formulation of the scalar conservation law is given by

>$$ \int_{K_j} \left( \partial_t c_{h,j} \right) \Phi_{j,m} \textrm{d}V + \oint_{\partial K_j} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \Phi_{j,m} \textrm{d}A - \int_{K_j} \vec{f}(c_{h,j})  \cdot \nabla \Phi_{j,m} \textrm{d}V = 0\quad \forall \Phi_{j,m} $$

- This system of equations has been discretized in space but not in time, hence it is called *semi-discrete*

- Commonly, we will abbreviate this as

>$$ \mathbf{M}_j \partial_t c_{h,j} + \oint_{\partial K_j} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \vec{\Phi}_{j}^{\textrm{T}} \textrm{d}A - \int_{K_j} \vec{f}(c_{h,j})  \cdot \nabla \vec{\Phi}_{j}^{\textrm{T}} \textrm{d}V = 0$$ 

using the mass matrix $\mathbf{M}_j$. This can be written even shorter as

>$$ \mathbf{M}_j \partial_t c_{h,j} + \mathbf{F}_{h,j} = 0 \quad \forall K_j$$ 

by defining the *discrete flux* or *discrete operator*

>$$ \mathbf{F}_{h,j}(c_h) :=  \oint_{\partial K_j} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \vec{\Phi}_{j}^{\textrm{T}} \textrm{d}A - \int_{K_j} \vec{f}(c_{h,j})  \cdot \nabla \vec{\Phi}_{j}^{\textrm{T}} \textrm{d}V $$ 

### 2.3 Incorporation of boundary conditions

- At boundary edges, $c_{h,j}^{+}(\vec{x}, t)$ is given by some boundary condition. For example, consider the scalar conservation law discussed above. On inflow boundaries, we *have* to
specify a boundary condition, while we *must not* define a boundary value at outflow boundaries.

- In any case, the definition for $c_{h,j}^{+}(\vec{x}, t)$ has to be extended in order to be valid on boundary edges. Thus, we define

>$$ c_{h,j}^{+}(\vec{x},t) = \begin{cases} \lim_{\epsilon \rightarrow 0^+} c_h(\vec{x} + \epsilon \vec{n}_{\partial K_j}, t) \quad \textrm{ if } \vec{x} \in \mathfrak{E}_h^{i} \\
c_B(\vec{x}, t)  \quad \textrm{ if } \vec{x} \in \mathfrak{E}_h^{b}
\end{cases} $$

with some boundary value $c_B(\vec{x}, t)$ that has to be specified

- If $c_B(\vec{x}, t)$ is a Dirichlet condition to be enforced on a particular edge, we thus simply insert it into the numerical flux function. Otherwise, i.e. if no boundary condition should be given, we set $c_B(\vec{x}, t) = c_{h,j}(\vec{x}, t)$.

## 3 Numerical fluxes

- In section 2.3, we have introduced the numerical flux function

>$$ \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \approx \vec{f}(c_h) \cdot \vec{n}_{\partial K_j} $$

that defines a unique flux across some inner edge $\mathcal{E}_e \in \mathfrak{E}^i_h$

- In principle, the following requirements only need to be satisfied on a set of *admissible states*.

- This chapter is largely based on (Di Pietro and Ern, 2012, Chapter 3.2)

### 3.1 Lipschitz continuity

- *Lipschitz continuity*: 
> $$ \exists C_a \in \mathbb{R}: \left| \hat f(a_1, b, \vec{n}) -  \hat f(a_2, b, \vec{n}) \right| < C_a \left| a_1 - a_2 \right| \quad \forall a_1, a_2 \in \mathbb{R} $$
and 
> $$ \exists C_b \in \mathbb{R}: \left| \hat f(a, b_1, \vec{n}) -  \hat f(a, b_2, \vec{n}) \right| < C_b \left| b_1 - b_2 \right| \quad \forall b_1, b_2 \in \mathbb{R} $$

- Lipschitz continuity is a technical assumption which is required to be able to prove the existence and uniqueness of a solution of the semi-discrete system.

- *Sufficient* condition: $ \hat f(a, b, \vec{n})$ is differentiable in $$ and $b$ and the magnitudes of the derivatives $ \partial_a \hat f(a, b, \vec{n})$ and  $ \partial_b \hat f(a, b, \vec{n})$ are bounded (within the set of admissible states)

### 3.2 Consistency

- *Consistency*:
>$$ \hat f(a,a,\vec{n}) = \vec{f}(a) \cdot \vec{n} \quad \forall a \in \mathbb{R}$$

- We can obtain a *strong form* of the semi-discrete formulation by integrating the volume by parts once again, but this time using only inner values (-) in the corresponding surface term:

>\begin{align} \int_{K_j} \left( \partial_t c_{h,j} \right) \Phi_{j,m} \textrm{d}V + \oint_{\partial K_j} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \Phi_{j,m} \textrm{d}A - \left( \oint_{\partial K_j} (\vec{f}(c_{h,j}^-) \cdot \vec{n}_{\partial K_j}) \Phi_{j,m} \textrm{d}A - \int_{K_j} (\nabla \cdot \vec{f}(c_{h,j})) \Phi_{j,m} \textrm{d}V \right) \\ 
= \int_{K_j} \left( \partial_t c_{h,j} \right) \Phi_{j,m} \textrm{d}V  + \int_{K_j} (\nabla \cdot \vec{f}(c_{h,j})) \Phi_{j,m} \textrm{d}V + \oint_{\partial K_j} \left( \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) - \vec{f}(c_{h,j}^-) \cdot \vec{n}_{\partial K_j} \right) \Phi_{j,m} \textrm{d}A = 0 
\end{align} 

- The strong form is rarely used in practice, but allows for an instructive interpretation of the role of the surface term: The first two terms are zero if $c_{h,j}$ is a *local* solution of the PDE, while the last term *weakly* enforces *flux continuity* across cell boundaries.

- In particular, if the consistency condition is fulfilled, it directly follows that the surface term vanishes if $c_h$ is *continuous* across cell boundaries $(c_{h,j}^- = c_{h,j}^+)$. In other words, the term is only *active* if the fluxes across edges are *not* continuous in order to enforce flux continuity

- As a consequence, we have 
>$$ \langle r(c_h) | \Phi_{j,m} \rangle_{K_j} 0 = \quad \forall \Phi_{j,m} $$
of $\hat f$ is consistent and $c_h$ is continuous and satisfies the PDE *locally*

### 3.3 Conservativity

- *Conservativity*:
>$$ \hat f(a,b,\vec{n}) = -\hat f(a,b,-\vec{n}) \quad \forall a,b \in \mathbb{R} $$

- The scalar conservation law is in conservative form. Integrating over the problem domain $\Omega$ leads to
>$$ \int_{\Omega} \partial_t c \textrm{d}V + \int_{\Omega} \nabla \cdot \vec{f}(c) \textrm{d}V = 0$$
and, assuming $c$ is smooth, further to
>$$ \partial_t \left(\int_{\Omega} c \textrm{d}V \right) + \oint_{\partial \Omega} \vec{f}(c) \cdot \vec{n}_{\partial \Omega} \textrm{d}A = 0 $$
after applying the Gauss theorem. We see that the total amount of $c$ only changes due to fluxes across
the domain boundary. In particular: If there is no flux across $\partial \Omega$, the total amount of $c$ is conserved for
all times, hence the term *conservative form*

- Taking w.l.o.g. $\Phi_{j,0} = 1$ in the semi-discrete weak formulation of the scalar conservation law gives
>$$ \int_{K_j} \left( \partial_t c_{h,j} \right) 1 \textrm{d}V + \oint_{\partial K_j} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) 1 \textrm{d}A - \underbrace{\int_{K_j} \vec{f}(c_{h,j})  \cdot \nabla 1 \textrm{d}V}_{=0} = 0\quad \forall K_j $$
which can be rewritten as
>$$ \partial_t \int_{K_j} c_{h,j} \textrm{d}V + \oint_{\partial K_j} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \textrm{d}A = 0\quad \forall K_j $$
which is the discrete *local* equivalent. However, this does *not* imply *global* conservativity.

- Summing over all elements gives
>$$ \sum_{K_j} \left( \partial_t \int_{K_j} c_{h,j} \textrm{d}V + \oint_{\partial K_j} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \textrm{d}A \right) = \partial_t \int_{\Omega} c_h \textrm{d}V + \sum_{K_j} \left( \sum_{\mathcal{E}\subset\partial K_j} \oint_{\mathcal{E}} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \textrm{d}A \right)$$

- All boundary integrals over inner edges $\mathcal{E} \in \mathfrak{E}_h^{i}$ appear exactly twice (i.e., once for each adjacent cell). Now consider a single edge $\mathcal{E} = \partial K_j \cap  \partial K_k$. Then conservativity of the flux leads to
>$$\begin{align} \int_{\mathcal{E}} \hat f(c_{h,j}^-, c_{h,j}^+, \vec{n}_{\partial K_j}) \textrm{d}A = -\int_{\mathcal{E}} \hat f(c_{h,j}^-, c_{h,j}^+, -\vec{n}_{\partial K_j}) \textrm{d}A \\
= -\int_{\mathcal{E}} \hat f(\tilde c_{h,j}^+, \tilde c_{h,j}^-, \vec{n}_{\partial K_k}) \textrm{d}A 
\end{align}$$
In other words, summing the contributions for an inner edge $\mathcal{E}$ (for $\Phi_{j,0} = 1$!) yields
>$$\begin{align} &\oint_{\mathcal{E}} \hat f(c_{h,j}^-, c_{h,j}^+, \vec{n}_{\partial K_j}) \textrm{d}A + \oint_{\mathcal{E}} \hat f(c_{h,j}^-, c_{h,j}^+, \vec{n}_{\partial K_j}) \textrm{d}A \\
&= \oint_{\mathcal{E}} \hat f(c_{h,j}^-, c_{h,j}^+, \vec{n}_{\partial K_j}) \textrm{d}A - \oint_{\mathcal{E}} \hat f(c_{h,j}^-, c_{h,j}^+, \vec{n}_{\partial K_j}) \textrm{d}A \textrm{d}A = 0,
\end{align}$$
which implies that all contributions from inner edges vanish. The only remaining edge contributions are thus given by the *boundary* edges $\mathcal{E}b$

- Consequently, (3.16) reduces to
>$$\begin{align} \partial_t \int_{\Omega} c_h \textrm{d}V + \sum_{K_j} \left( \sum_{\mathcal{E}\subset\partial K_j} \oint_{\mathcal{E}} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \textrm{d}A \right) \\ 
= \partial_t \int_{\Omega} c_h \textrm{d}V + \sum_{\mathcal{E}\ni\mathfrak{E}_h^b} \int_{\mathcal{E}} \hat f(c_{h,j}^{-}, c_{h,j}^{+}, \vec{n}_{\partial K_j}) \textrm{d}A  \\
\partial_t \int_{\Omega} c_h \textrm{d}V + \oint_{\partial \Omega} \hat f(c_{h}^{-}, c_B, \vec{n}_{\partial K_j}) \textrm{d}A  \end{align}$$
which is a discrete equivalent of (3.11) and hence shows *global* conservativity.

### 3.4 Monotonicity

- *Monotonicity*
>$$ \frac{\partial \hat f(a,b,\vec{n})}{\partial a} \geq 0 \wedge \frac{\partial \hat f(a,b,\vec{n})}{\partial b} \leq 0 \quad \forall a,b,\vec{n}  $$

- Interpretation: *Monotonicity* in combination with *consistency* can be visualized as follows

<img src="images/Monotonicity.PNG" style="width: 600px;"/>

- Monotonicity is required to prove *stability* of a scheme. Note that there are several notions of stability in numerical methods. In this section, we define stability in the continuous setting via the *energy estimate*
>$$ \lVert c(x,t) \rVert_{\Omega}^2 \leq \lVert c(x,0) \rVert_{\Omega}^2 \quad \forall t \geq 0 $$
where we assume zero Dirichlet boundary conditions wherever applicable. That is, we call a system stable if the *energy norm* $\lVert c(x,t) \rVert_{\Omega}^2$ of the system can only decrease in the absence of inflow

- The discrete equivalent is the energy estimate
>$$ \lVert c_h(x,t) \rVert_{\Omega}^2 \leq \lVert \pi_k (c_h(x,0)) \rVert_{\Omega}^2 \quad \forall t \geq 0 $$
where we once again assume zero Dirichlet boundary conditions wherever applicable

- Remark: The quantity
>$$ \Theta = \frac{1}{2} \lVert \pi_k (c_h(x,0)) \rVert_{\Omega}^2 - \frac{1}{2} \lVert c_h(x,0) \rVert_{\Omega}^2 $$
is a measure for the numerical dissipation of a scheme. It must be positive for all times, but should be as low as possible too.

### Hands-on 2

## 4 Implementation issues

### 4.1 Evaluation of DG-representations

**Basis functions defined on reference elements**

- Define reference elements. E.g. in the BoSSS code, we use
    - unit line $K_{\textrm{line}}^{\textrm{ref}} = (-1,1)$
    - unit square $K_{\textrm{square}}^{\textrm{ref}} = (-1,1)^2$
    - unit cube $K_{\textrm{cube}}^{\textrm{ref}} = (-1,1)^2$

- Each cell $K_j$ is the image of a reference element

<img src="images/ReferenceElementSquare.PNG" style="width: 600px;"/>

- Coordinate transformation
>$$ \vec{\xi} \mapsto \vec{x} = T_j(\vec{\xi}) \quad \vec{xi} = T_j^{-1}(\vec{x})$$

- The transformation $T_j$ may be either affine-linear (in this case, $K_j$ is called a *linear* cell) or nonlinear ($K_j$ is nonlinear or *curved* cell)

- For each reference element, we define a list of polynomials, e.g.
>$$ \vec{\Phi}^{\textrm{ref}} := (\Phi_0^{\textrm{ref}}, \Phi_1^{\textrm{ref}}, \Phi_2^{\textrm{ref}}, \dots)$$
which are orthonormal on the reference element, i.e.
>$$ (\Phi_n^{\textrm{ref}}, \Phi_m^{\textrm{ref}})_{\vec{\xi}} = \int_{K^{\textrm{ref}}} \Phi_n(\vec{\xi}) \Phi_m(\vec{\xi}) \textrm{d}V = \delta_{nm} $$

- The basis in physical coordinates, in cell $K_j$,
>$$ \vec{\Phi}_{j,-} = ( \Phi_{j,0}, \dots, \Phi_{j,N_k - 1})$$
is obtained via a transformation of a *upper-triangular* matrix $B \in \mathbb{R}^{N_k \times N_k}$
>$$ \vec{\Phi}_{j,-} =  \vec{\Phi}^{\textrm{ref}} B_{j,-}$$
>$$ \Phi_{j,m} = \begin{cases} \sum_{n=0}^{N_k - 1} \Phi_n^{\textrm{ref}} (T^-1(\vec{x})) B_{j,mn} \quad \textrm{ if } \vec{x} \in K_j \\
0 \quad \textrm{elsewhere}\end{cases} $$
So that $\Phi_{j,m}$ are orthonormal in the physical space, i.e.
>$$ (\Phi_{j,m}, \Phi_{l,n}) = \int_{\Omega} \Phi_{j,m}(\vec{x}) \Phi_{j,m}(\vec{x}) \textrm{d}V = \delta_{jl} \delta_{mn} $$ 

- The orthogonality in indices $l$, $j$ is trivial, since $\Phi_{j,m}$ is zero in cell $K_l$, for $j \neq l$.

- To get orthonormality in the case $j = l$, we have to consider the integral transformation:
>$$ \int_{\vec{x} \in K_j} f(\vec{x}) \textrm{d}V = \int_{\vec{\xi} \in K^{\textrm{ref}}} f (T_j(\vec{\xi})) \cdot \det((\partial_{\vec{\xi}} T_j)(\vec{\xi})) \textrm{d}V$$

- for linear cells, we can represent the transformation $T_j$ as
>$$ T_j(\vec{\xi}) = \mathbf{T}_j \vec{\xi} + b_j$$
thus the Jacobian $\partial_{\vec{}\xi}T$ is constant and therefore we can define
>$$ J_j := \det(\mathbf{T}_j)$$
and choosing
>$$ B_{j,nm} = \begin{cases} \frac{1}{\sqrt{J_j}} \quad \textrm{ for } n=m \\ 0 \quad \textrm{elsewhere}\end{cases}$$
yields orthogonality an cell $K_j$:
>$$ (\Phi_{j,n}, \Phi_{j,m}) = \int_{K_j} \Phi_{j,n} \Phi_{j,m} \textrm{d}V = \frac{1}{\sqrt{J_j}} \frac{1}{\sqrt{J_j}} J_j \int_{K^{\textrm{ref}}} \Phi_{n}^{\textrm{ref}} \Phi_{m}^{\textrm{ref}} \textrm{d}V$$
Therefore, in a linear cell $K_j$, it is not necessary to store the whole matrix $B_{j,−}$, but only number $J_j$ resp. $\frac{1}{\sqrt{J_j}} $.
We see that in linear cells, *orthogonality* is preserved under the transformation ($n \neq m$, then $\int_{K_j} \Phi_{j,n} \Phi_{j,m} \textrm{d}V = 0$ because $\int_{K^{\textrm{ref}}} \Phi_{n}^{\textrm{ref}} \Phi_{m}^{\textrm{ref}} \textrm{d}V$ is zero), but *orthonormality* ($(\Phi_{j,n}\Phi_{j,m}) = 1$) is in general lost.

**Evaluation** We want to evaluate $u(\vec{x}) = \sum_{jn} \Phi_{j,n} (\vec{x}) \tilde u_{j,n}$

- For a single evaluation at one point x, there is not much optimization potential form an algorithm point-of-view.

- If we evaluate in *all cells* and use *the same nodes in reference coordinates* in those cells, i.e. we have nodes
>$$ \vec{\xi}_0, \dots, \vec{\xi}_{K-1} \textrm{ and } x_{j,k} := T_j(\vec{\xi}_k)\$$
We can perform some algorithmic optimization in evaluating
>$$ u_{j,k} := u(x_{j,k}) $$
To evaluate $u_{j,k}$, we have to compute
>$$ \begin{align} \forall j,k: u_{j,k} &= \sum_{n=0}^{N_k-1} \Phi_{j,n}(\vec{x}_k) \tilde u_{j,n} \\
&= \sum_{n}\left( \sum_{l=0}^{N_k-1} \Phi_{l}^{\textrm{ref}} B_{j,ln} \right) \tilde u_{j,n} \quad \implies \mathcal{O}(J K N_k^2)\\
&= \sum_{l=0}^{N_k-1} \underbrace{\left( \sum_{n=0}^{N_k-1} B_{j,ln} \tilde u_{j,n} \right)}_{=: \tilde v_{j,l}} \underbrace{\Phi_{l}^{\textrm{ref}}}_{\Phi_{l,k}^{\textrm{ref}}} \quad \implies \mathcal{O}(J K N_k) \textrm{ or } \mathcal{O}(J N_k^2)\\
&= \sum_{n=0}^{N_k-1} \left( \frac{1}{\sqrt{J_j}} \tilde u_{j,n} \right) \Phi_{l,k}^{\textrm{ref}} \\
\end{align}$$

### 4.2 Quadrature

- approximation of integration: nodes $(\xi_0, \dots , \xi_{K−1}) =: \vec{\xi}$, weights $(w_0, \dots ,w_{K-1}) =: \vec{w}$
>$$ \int_{K^{\textrm{ref}}} f(\vec{\xi}) \textrm{d}V \approx \sum_{k=0}^{K-1} f(\xi_k)w_k := \int_{(\vec{\xi},\vec{w})}^{\textrm{num}} f $$

- the rule $(\vec{\xi},\vec{w})$ has order $p$, if for all polynomials $f(\vec{\xi})$ with degree $f \leq g$ we have $\int_{(\vec{\xi},\vec{w})}^{\textrm{num}} f = \int_{K^{\textrm{ref}}} f(\vec{\xi}) \textrm{d}V $.

- In 1D: Gauss rules are most efficient: $p = 2K − 1$

**Rules for square and cube elements** For $K_{\textrm{square}}^{\textrm{ref}}, K_{\textrm{cube}}^{\textrm{ref}}$, tensorized rules: e.g., assume 1D-rule $(\vec{\xi}^{\textrm{line}},\vec{w}^{\textrm{line}})$ of order $p$, with $K$ nodes, then we have a rule $(\vec{\xi}^{\textrm{square}},\vec{w}^{\textrm{square}})$ for $K_{\textrm{square}}^{\textrm{ref}}$ with $
\vec{\xi}_{lK+k}^{\textrm{square}} = ({\xi}_l^{\textrm{line}},{\xi}_k^{\textrm{line}}), w_{lK+k}^{\textrm{square}} = ({w}_l^{\textrm{line}},{w}_k^{\textrm{line}})$
So, the efficient 1D-Gauss rules can also be used in 2D resp. 3D: $p = 2\sqrt[D]{K} - 1$.

## 5 Linear, scalar equations of second order

**Prototype problem: Poisson equation** domain $\Omega \subseteq \mathbb{R}^D, D=1,2,3$, boundary of $\Omega : \partial \Omega = \Gamma_D \cup \Gamma_N$
>$$ \begin{cases} - \Delta u &= g_{\Omega} \quad \textrm{ in } \Omega \\
u &= g_D \quad \textrm{ on } \Gamma_D \\
\nabla u \cdot \vec{n}_{\partial \Omega} &= g_N \quad \textrm{ on } \Gamma_N \end{cases} $$

### 5.1 Variational formulation of the Poisson equation, continuous setting

Idea: multiply Poisson equation with test function, and perform partial integration of $\Omega$. To handle boundary
conditions, we define a function space that only admits functions $u$ with $u|_{\partial \Omega} = 0$. (Sobolev-spaces, idea from
Sergej Sobolev, est. 1950)

**Definition (Sobolev Space):**
>$$ H_0^1(\Omega) := \{ u \in L^2(\Omega); \| u\|_2 + \| \nabla u\|_2 < \infty; u|_{\partial \Omega} = 0 \}. $$

**Weak formulation of the Poisson problem:** Regarding the Poisson problem, we search for a weak solution $u$ in the Sobolev-space $H_0^1(\Omega)$: Find $u \in H_0^1(\Omega)$ so that
>$$ a(u,v) = \int_{\Omega} g_{\Omega} v \ \textrm{d}V \quad \forall v \in H_0^1(\Omega) $$
with
>$$ a(u,v) := \int_{\Omega} \nabla u \cdot \nabla v \ \textrm{d}V.$$


### 5.2 Global variational formulation, discrete setting

We start with a generic, time-independent equation
>$$ \textrm{div}(\vec{f}(u)) = g_{\Omega} $$

As in Section 2.2, we multiply by a test function $\Phi_{jm}$, integrate over $K_j$, apply integration-by-parts and introduce
a numerical flux $\hat F$ to obtain
>$$ \oint_{\partial K_j} \hat F (u_j^-, u_j^+, \vec{n}_{\partial K_j}) \Phi_{jm} \ \textrm{d}A - \int_{K_j} \vec{f}(u_j) \cdot \nabla \Phi_{jm} \ \textrm{d}V = \int_{K_j} g_{\Omega} \Phi_{jm} \ \textrm{d}V $$
This is a cell-by-cell-formulation. To obtain a global formulation we have to sum over all cells.

In addition, we replace the test-function $\Phi_{jm}$ with an arbitrary test function $v$ with the representation $v = \sum_j \sum_m \tilde v_{jm} \Phi_{jm}$. Therefore, we also multiply the above equation with a constant $\tilde v_{jm}$ and sum over $m$.

Right-hand-side:
>$$ \sum_j \sum_m \tilde v_{jm} \int_{K_j} g_{\Omega} \Phi_{jm} \ \textrm{d}V = \int_{\Omega} g_{\Omega} \left( \sum_j \sum_m \tilde v_{jm} \Phi_{jm}\right) \ \textrm{d}V = \int_{\Omega} g_{\Omega} v \ \textrm{d}V $$

Volume part (left-hand-side):

**Definition: (broken gradient $\nabla_h$)** For some function $f$ which is sufficiently smooth within the cells of $\mathfrak{K}_h$,
but may be discontinuous at the cell boundaries, i.e.
>$$ f \in C^1 (\Omega \setminus (\cup_j \partial K_j))\$$
we define
>$$ \nabla_h f = \begin{cases} 0 \quad \textrm{ on } \cup_j \partial K_j \\ \nabla f \quad \textrm{ elsewhere }\end{cases}$$
To keep notation simple, we may omit the h-index of the broken gradient ∇h in all weak formulations.

Using the broken gradient, we can take the sum of the volume part over all $j$ and $m$ to obtain
>$$ \sum_j \sum_m \tilde v_{jm} \int_{K_j} \vec{f}(u_j) \cdot \nabla \Phi_{jm} \ \textrm{d}V = \int_{\Omega} \vec{f}(u) \left( \sum_j \sum_m \tilde v_{jm} \nabla_h \Phi_{jm}\right) \ \textrm{d}V = \int_{\Omega} \vec{f}(u) \nabla_h v \ \textrm{d}V $$

Surface part:

- The sets of all edges $\Gamma$ and all inner edges $\Gamma_i$
>$$ \Gamma :=  \cup_j \partial K_j \textrm{ and } \Gamma_i = \Gamma \setminus \partial \Omega$$

- a normal field on $\Gamma$, with $\vec{n}_\Gamma$: on $\partial \Omega$, $\vec{n}_{\Gamma} = \vec{n}_{\partial \Gamma}$; on $\Gamma_i$, we have to pick one normal (out of two options) For each edge $\mathcal{E} \subseteq \Gamma$, we therefore have an “in” and an “out” – cell.

- We re-define the limits $u^+$ and $u^-$ (Section 2.2) with respect to \Gamma:
>$$ u^- = \lim_{\epsilon \rightarrow 0, \epsilon > 0} (u(\vec{x} - \vec{n}_{\Gamma} \epsilon))$$
>$$ u^+ = \lim_{\epsilon \rightarrow 0, \epsilon > 0} (u(\vec{x} + \vec{n}_{\Gamma} \epsilon))$$

Note that on $\Gamma_i$, the value of the DG-Fields is *not* unique; the limit on the in-side and on the out-side are usually different.

- This re-definition of in- and out-values also affects the average- and the jump-operator. We also define jump and average on the boundary:
>$$ \textrm{on } \Gamma_i: \llbracket u \rrbracket := (u^- - u^+) \quad \{ u \} := \frac{1}{2}(u^- + u^+)$$
>$$ \textrm{on } \partial \Omega: \llbracket u \rrbracket := u^- \quad \{ u \} := u^-$$

- Illustration for two cells:

<img src="images/weakFormulation2Cells.PNG" style="width: 600px;"/>

- Note that
>$$ \hat F(a,b,\vec{n}_{\partial K}) = \begin{cases} \hat F(a,b,\vec{n}_\Gamma) \quad \vec{n}_{\partial K}  = \vec{n}_{\Gamma} \\ -\hat F(a,b,\vec{n}_\Gamma) \quad \vec{n}_{\partial K}  = -\vec{n}_{\Gamma}\end{cases}$$

- For any discontinuous function v, we have
>$$ \sum_j \oint_{\partial K_j} v \vec{n}_{\partial K_j} \ \textrm{d}A = \oint_{\Gamma} \llbracket v \rrbracket \vec{n}_{\Gamma} \ \textrm{d}A $$

Note that this is independent of the choice on $\vec{n}_{\Gamma}$. Finally, we are ready to sum up the surface part of the
left-hand side:
>$$ \sum_j \sum_m \tilde v_{jm} \oint_{\partial K_j} \hat F \Phi_{jm} \ \textrm{d}A = \oint_{\Gamma} \hat F  \llbracket v \rrbracket \ \textrm{d}A $$

combining all above, we obtain
>$$ \underbrace{\oint_{\Gamma} \hat F (u^-, u^+, \vec{n}_{\Gamma}) \llbracket v \rrbracket \ \textrm{d}A  - \int_{\Omega} \vec{f}(u) \nabla_h v \ \textrm{d}V}_{=: a_h(u,v) } = \int_{\Omega} g_{\Omega} v \ \textrm{d}V$$

Thus the global variational formulation reads: find $u \in \mathbb{P}_k(\mathfrak{K}_h)$ so that
>$$ a_h(u,v) = \int g_{\Omega} v \ \textrm{d}V \quad \forall v \in \mathbb{P}_k(\mathfrak{K}_h)$$

### 5.3 The Lax-Milgram theorem

**Definition: coercivity** A bilinear form $a(-,-)$ is coercive, if there is a constatn $\gamma > 0$ so that in *some suitable norm* $\| - \|_\ast$ the inequality
>$$ a(u,u) \geq \gamma \| u \|_\ast^2 \quad \forall u $$
holds.

**Theorem: Lax-Milgram** If a bilinear form $a(-,-)$ is coercive, the solution to the variational problem ($a(u,v) = \int g_\Omega v \ \textrm{d}V \ \forall v$) is unique

### 5.4 Symmetric Interior Penalty (SIP)

One possibility for a Discretization that is *stable* and *consistent*. (Alternatives: Local non-symmetric interior penalty.)
>$$ a_{\textrm{SIP}}(u,v) = \int_{\Omega} \underbrace{\nabla u \cdot \nabla v}_{\textrm{volume term}} \ \textrm{d}V - \oint_{\Gamma \setminus \Gamma_N} \underbrace{\{ \nabla u\} \cdot \vec{n}_\Gamma \llbracket v \rrbracket}_{\textrm{consistency term}} + \underbrace{\{ \nabla v\} \cdot \vec{n}_\Gamma \llbracket u \rrbracket}_{\textrm{symmetry term}} \ \textrm{d}A + \oint_{\Gamma \setminus \Gamma_N} \underbrace{\eta \llbracket u \rrbracket \llbracket v \rrbracket}_{\textrm{penalty term}} \ \textrm{d}A $$

- The penalty prevents spurious solutions

- The penalty factor $\eta(\vec{x})$ must be chosen large enough, otherwise the method is unstable; in each cell, one might
pick
>$$ \eta = c \cdot k^2 \frac{| \partial K |}{|K|}, \quad c \approx 1$$
and on $\Gamma$
>$$ \eta = \max(\eta^-, \eta^+)$$

- $a_{\textrm{SIP}}(-,-)$ is symmetric: reflects symmetry of $a(−, −)$ and is good for solvers.

**Stability of the SIP-form:** According to the Lax-Milgram-theorem, we have to show coercivity of the SIP-form $a_{\textrm{SIP}}(−,−)$, i.e. we have to show that there is a constant so that
>$$ a_{\textrm{SIP}}(u,u) \geq \gamma \| u \|_{\textrm{SIP}}^2 \quad \forall u \in \mathbb{P}_k(\mathfrak{K}_h)$$
with the so-called SIP-norm
>$$ \| u \|_{\textrm{SIP}} := \left( \| \nabla u\|_{L^2(\Omega)}^2 + \oint_{\Gamma} \frac{1}{h_e} \llbracket u \rrbracket^2 \ \textrm{d}A \right)^{\frac{1}{2}}$$
Wee see that (for $\Gamma_N = \emptyset$)
>$$ a_{\textrm{SIP}}(u,u) = \underbrace{\int_{\Omega} \nabla u \cdot \nabla v \ \textrm{d}V}_{= \| \nabla u\|_{L^2(\Omega)}^2} - \oint_{\Gamma} \{ \nabla u\} \cdot \vec{n}_\Gamma \llbracket v \rrbracket + \{ \nabla v\} \cdot \vec{n}_\Gamma \llbracket u \rrbracket \ \textrm{d}A + \oint_{\Gamma} \eta \llbracket u \rrbracket \llbracket v \rrbracket \ \textrm{d}A $$
and the inequality reduces to
>$$ \| \nabla u\|_{L^2(\Omega)}^2 - 2 \oint_{\Gamma} \{ \nabla u\} \cdot \vec{n}_\Gamma \llbracket u \rrbracket \ \textrm{d}A + \oint_{\Gamma} \eta \llbracket u \rrbracket^2 \ \textrm{d}A \geq \gamma \| \nabla u\|_{L^2(\Omega)}^2 + \gamma \oint_{\Gamma} \frac{1}{h_e} \llbracket u \rrbracket^2 \ \textrm{d}A$$ 
The critical part is
>$$ (1- \gamma) \| \nabla u\|_{L^2(\Omega)}^2 - 2 \oint_{\Gamma} \{ \nabla u\} \cdot \vec{n}_\Gamma \llbracket u \rrbracket \ \textrm{d}A + \oint_{\Gamma} \left(\eta - \frac{\gamma}{h_e}\right) \llbracket u \rrbracket^2 \ \textrm{d}A \geq 0$$ 
The basic idea of the proof (of coercivity of the SIP-norm) is to show that if $η$ is large enough, the above inequality is fulfilled for every $u$ in the DG-space. This is sometimes called: “obtaining control over the gradients by choosing a sufficiently large penalty”. The actual proof is rather technical and will be skipped, see e.g. Hillewaert (2013) or Di Pietro and Ern (2012).

### 5.5 Poisson equation as a system

Obviously, it is also possible to discretize the Poisson equation as a system of first-order-PDE’s, introducing a vector field $\vec{\sigma}$:
>$$ \begin{align} \vec{\sigma} + \nabla u &= 0, \quad \textrm{ in } \Omega \\ 
\textrm{div}(\vec{\sigma}) &= g_\Omega, \quad \textrm{ in } \Omega \\ 
u &= g_{\textrm{D}}, \quad \textrm{ on } \Gamma_{\textrm{D}} \\
-\vec{\sigma} \cdot \vec{n}_{\partial \Omega} &= g_{\textrm{N}}, \quad \textrm{ on } \Gamma_{\textrm{N}} \end{align}$$
resp. in matrix-notation:
>$$ \begin{bmatrix} \mathbf{1} & \nabla \\ \textrm{div} & 0\end{bmatrix} \cdot \begin{bmatrix} \vec{\sigma} \\ u \end{bmatrix} = \begin{bmatrix} 0 \\ g_\Omega \end{bmatrix}$$

**DG-Discretization of the Poisson system** Multiplying with a test function $\vec{\tau}$ and integrating over a cell $K_j$
>$$ \int_{K_j} \vec{\sigma} \cdot \vec{\tau} \ \textrm{d}V + \int_{K_j} \nabla u \cdot \vec{\tau} \ \textrm{d}V = 0 $$
Integrating by parts:
>$$ \int_{K_j} \vec{\sigma} \cdot \vec{\tau} \ \textrm{d}V +  \oint_{\partial K_j} u \ \vec{n}_{\partial K_j} \cdot \vec{\tau} \ \textrm{d}A - \int_{K_j} \textrm{div}(\vec{\tau}) u \ \textrm{d}V = 0 $$
We replace $u$ in the surface integral with
>$$ u = \begin{cases} \{u\} \quad \textrm{ on } \Gamma_i \\ 
g_{\textrm{D}} \quad \textrm{ on } \Gamma_{\textrm{D}} \\ 
\{u\} = \{u^-\} \quad \textrm{ on } \Gamma_{\textrm{N}} \\ \end{cases}$$
and sum over all $K_j$ in the grid $\mathfrak{K}_h$
>$$ \int_{\Omega} \vec{\sigma} \cdot \vec{\tau} \ \textrm{d}V + \underbrace{\oint_{\Gamma \setminus \Gamma_{\textrm{D}}} \{ u \} \llbracket \vec{\tau} \rrbracket \cdot \vec{n}_{\Gamma} \ \textrm{d}A - \int_{\Omega} \textrm{div}(\vec{\tau}) u \ \textrm{d}V}_{=:b(u,\vec{\tau})} = \underbrace{ \oint_{\Gamma_{\textrm{D}}} -g_{\textrm{D}} \llbracket \vec{\tau} \rrbracket \cdot \vec{n}_{\Gamma} \ \textrm{d}A}_{=: r_{\textrm{D}}(\vec{\tau})} $$

In the same fashion, we derive a weak form for the second equation. Multiply by test function $v$, integrate over $K_j$
>$$ \int_{K_j} \textrm{div}(\vec{\sigma}) v \ \textrm{d}V = \int_{K_j} g_\Omega v \ \textrm{d}V $$
apply integration by parts, use the central-differnece flux
>$$ \hat{\vec{\sigma}} \cdot \vec{n} = \begin{cases} \{\vec{\sigma}\} \cdot \vec{n} \quad \textrm{ on } \Gamma_i \\ 
\vec{\sigma}^- \cdot \vec{n} \quad \textrm{ on } \Gamma_{\textrm{D}} \\ 
g_{\textrm{N}} \quad \textrm{ on } \Gamma_{\textrm{N}} \\ \end{cases}$$
and sum over all cells
>$$ \underbrace{\oint_{\Gamma \setminus \Gamma_{\textrm{N}}} \{ \vec{\sigma} \cdot \vec{n}_\Gamma \} \llbracket v \rrbracket \ \textrm{d}A - \int_{\Omega} \vec{\sigma} \nabla v \ \textrm{d}V  }_{=:c(\vec{\sigma}, v)} = \int_{\Omega} g_\Omega v \ \textrm{d}V + \underbrace{\oint_{\Gamma_{\textrm{N}}} - g_{\textrm{N}} v \ \textrm{d}A}_{=: r_{\textrm{N}}(v)}$$

To discretize $\vec{\sigma}$ and $u$, we use DG-spaces of order $k$ and $l$. Finally we arrive at the following DG-discretization:
Find $\vec{\sigma} \in \mathbb{P}_k(\mathfrak{K}_h)^D$ and $u \in \mathbb{P}_l(\mathfrak{K}_h)$ so that
>$$ \begin{align} \int_{\Omega} \vec{\sigma} \cdot \vec{\tau} \ \textrm{d}V + b(u,\vec{\tau}) &= r_{\textrm{D}}(\vec{\tau}) \quad \forall \vec{\tau} \in \mathbb{P}_k(\mathfrak{K}_h)^D \\ c(\vec{\sigma}, v) &= \int_{\Omega} g_\Omega v \ \textrm{d}V + r_{\textrm{N}}(v) \quad \forall v \in \mathbb{P}_l(\mathfrak{K}_h) \end{align}$$

**Symmetry of the formulation** One can show that $b(u, \vec{\tau}) = -c(\vec{\tau}, u)$
Using this identity, one can sum both equations above, to obtain the following representation: Find $\vec{\sigma} \in \mathbb{P}_k(\mathfrak{K}_h)^D$ and $u \in \mathbb{P}_l(\mathfrak{K}_h)$ so that
>$$ \underbrace{\int_{\Omega} \vec{\sigma} \cdot \vec{\tau} \ \textrm{d}V + b(u,\vec{\tau}) - b(v,\vec{\sigma})}_{=: B((\vec{\sigma}, u),(\vec{\tau}, v))} = \int_{\Omega} g_\Omega v \ \textrm{d}V + r_{\textrm{D}}(\vec{\tau}) + r_{\textrm{N}}(v) \quad \forall (\vec{\tau},v) \in \mathbb{P}_k(\mathfrak{K}_h)^D \times \mathbb{P}_l(\mathfrak{K}_h) $$

**Stability of the system-formulation**

Unfortunately, the form $B(−, −)$ is not coercive; it is only partially coercive:
>$$ B((\vec{\sigma}, v),(\vec{\sigma}, v)) = \int_{\Omega} \vec{\sigma} \cdot \vec{\sigma} \ \textrm{d}V + b(v,\vec{\sigma}) - b(v, \vec{\sigma}) = \| \vec{\sigma} \|_{L^2(\Omega)}^2$$
It is obvious that there exists a coercivity constant $\gamma > 0$ so that
>$$ B((\vec{\sigma}, v),(\vec{\sigma}, v)) \geq \| \vec{\sigma} \|^2 $$
However, we fail to realize
>$$ B((\vec{\sigma}, v),(\vec{\sigma}, v)) \geq \| (\vec{\sigma},v) \|_\ast^2 $$
since e.g. for any $v$ with $\| (0,v)\|_∗ > 0$
>$$ B((0, v),(0, v)) = 0 < \| (0,v) \|_\ast^2$$
Therefore, the Lax-Milgram-theorem does not apply. This does not necessarily mean that the Problem is ill-posed. 
It is, however still possible to prove that the formulation (7.15) coercivity resp. is a sufficient condition, but it is a necessary condition.

**Theorem: Banach-Nečas-Babuška (BNB), inf-sup-condition** The variational problem:
>$$ a(u,v) = \int f \ v \ \textrm{d}V \quad \forall v$$
has a unique solution (is well-posed) if, and only if such that
1. there exists a constant $C_{\textrm{Sta}} > 0$ so that for all $v$
>$$ C_{\textrm{Sta}} \| v \| \leq \sup_{w \neq 0} \frac{a(v,w) }{\| w \|_\ast}$$
2. if $a(v, w) = 0 \forall v$, this implies that $w = 0$.

Since the first condition can be reformulated as
>$$ C_{\textrm{Sta}} \| v \| \leq \inf_{v \neq 0} \sup_{w \neq 0} \frac{a(v,w) }{\| v \|_\ast \| w \|_\ast}$$
this is also called *inf-sup* condition

**Remark: (saddle-point problems)** Systems like
>$$ \begin{align} a(\vec{u},\vec{v}) + b(p,\vec{v}) = f_1(\vec{v}) \quad \forall \vec{v} \\ - b(q,\vec{u}) = f_2(q) \quad \forall q \end{align}$$ 
if discretized lead to matrices of the form
>$$ \begin{bmatrix} A & B \\ -B^{\textrm{T}} & 0\end{bmatrix}$$
Such matrices are very often called saddle-point matrices. The actual definition of a saddle-point matrix is that
it has Eigenvalues with positive and negative sign. For coercive problems, on the other hand, all Eigenvalues
have the same sign.
- Since the saddle-point system is reminiscent of a Stokes-problem, we call variables $u, v, w$ velocities, which are members of a velocity space. 
Analogously, the pressure-type variables $p, q, r$ are members of a pressure space.
- The most critical part is the stability of the pressure. Showing such an estimate usually is the most difficult part of the proof. 
It heavily depends on the choice for velocity and pressure space and on the form a(−, −)

### Hands-on 3

## 6 Incompressible flows

**Equations and boundary conditions, strong form:** Steady Stokes, note similarity to the Poisson equation

>$$ \begin{align} - \frac{1}{\textrm{Re}} \Delta \vec{u} + \nabla p &= \vec{g}_{\Omega} \\
\textrm{div}(\vec{u}) &= 0 \end{align}$$

Steady Navier-Stokes:
>$$ \begin{align}\textrm{div}(\vec{u} \otimes \vec{u}) - \frac{1}{\textrm{Re}} \Delta \vec{u} + \nabla p &= \vec{g}_{\Omega} \\
\textrm{div}(\vec{u}) &= 0 \end{align}$$

Instationary Navier-Stokes:
>$$ \begin{align} \partial_t \vec{u} + \textrm{div}(\vec{u} \otimes \vec{u}) - \frac{1}{\textrm{Re}} \Delta \vec{u} + \nabla p &= \vec{g}_{\Omega} \\
\textrm{div}(\vec{u}) &= 0 \end{align}$$

Boundary conditions:
>$$  \begin{align} \vec{u} &= \vec{u}_{\textrm{D}} \quad \textrm{ on } \Gamma_{\textrm{D}} \\
\left( -\frac{1}{\textrm{Re}} \nabla \vec{u} + p \right) \vec{n}_{\partial \Omega} &= 0 \quad \textrm{ on } \Gamma_{\textrm{N}} \end{align}$$

If we have only Dirichlet boundary conditions, the pressure $p$ is only unique up to a constant value, i.e. if $(\vec{u}, p)$ is a solution for the Stokes/Navier-Stokes equation, so is ($u$, $p$ + const.). An additional condition is required
to fix the pressure, e.g. $p(\vec{x}_0) = 0$ or $\int_{\Omega} p \textrm{d}V = 0$

### 6.1 Steady Stokes, discrete setting

We reuse the SIP and one has to fulfill the inf-sup condition in order to guarantee well-posedness of the problem. In the
context of the Stokes-equation, this is also known as the LBB-condition (*Ladyzhenskaya-Babuška-Brezzi*).

**Mixed order discretizations**

- The polynomial degree of velocity is $k$, for pressure it is $k − 1$.
- Stability is only known/proven for triangular grids, without hanging nodes, $k ≤ 3$, see Girault et al. (2005).
- Seems to work also on other grids and for higher orders.
- We search for $(\vec{u}, p) \in \mathbb{P}_k(\mathfrak{K}_h)^D \times \mathbb{P}_{k-1}(\mathfrak{K}_h)$, so that
>$$ \begin{align} a(\vec{u}, \vec{v}) + b(p, \vec{v}) &= \int_{\Omega} \vec{f} \cdot \vec{v} \ \textrm{d}V \quad \forall \vec{v} \in \mathbb{P}_{k}(\mathfrak{K}_h)^D \\
-b(\tau, \vec{u}) &= 0 \quad \forall \tau \in \mathbb{P}_{k-1}(\mathfrak{K}_h) \end{align}$$

- Definition of diffusive terms:
>$$ a(\vec{u}, \vec{v}) := \frac{1}{\textrm{Re}}\sum_d a_{\textrm{SIP}}(u_d, v_d)$$

**Equal order discretization**

- The same polynomial order $k$ for velocity and pressure.
- Requires an additional stabilization term, stability proof Di Pietro and Ern (2012)
- We search for $(\vec{u}, p) \in \mathbb{P}_k(\mathfrak{K}_h)^D \times \mathbb{P}_{k-1}(\mathfrak{K}_h)$, so that
>$$ \begin{align} a(\vec{u}, \vec{v}) + b(p, \vec{v}) &= \int_{\Omega} \vec{f} \cdot \vec{v} \ \textrm{d}V \quad \forall \vec{v} \in \mathbb{P}_{k}(\mathfrak{K}_h)^D \\
-b(\tau, \vec{u}) + \textrm{Re} \ s(p,\tau) &= 0 \quad \forall \tau \in \mathbb{P}_{k-1}(\mathfrak{K}_h) \end{align}$$
where the pressure stabilization term is defined as
>$$ s(p,\tau) := \oint_{\Gamma_i} \eta_s \llbracket p \rrbracket \llbracket \tau \rrbracket \ \textrm{d}A$$
and $\eta_s$ equal to the area of the respective cell-face.

### 6.2 Steady Navier-Stokes, discrete setting

**Discretization of the convective part:** We need an approximation to the trilinear form $t(−,−,−)$.

- Skew- Symmetric form, see Di Pietro and Ern (2012)
>$$ t(\vec{w}, \vec{u}, \vec{v}) = \int_{\Omega} (\vec{w} \cdot \nabla \vec{v}) \cdot \vec{u} + \frac{1}{2} \textrm{div}(\vec{w})(\vec{v} \cdot \vec{u}) \ \textrm{d}V + \oint_{\Gamma \setminus \Gamma_{\textrm{D}}} (\{ \vec{w}\} \cdot \vec{n}_\Gamma) (\llbracket \vec{v} \rrbracket \cdot \{ \vec{u}\} ) \ \textrm{d}A - \oint_{\Gamma} \frac{1}{2} \llbracket \vec{w} \rrbracket \cdot \vec{n}_{\Gamma} \{ \vec{u} \cdot \vec{v} \} \ \textrm{d}A  $$

- Flux-based form:
>$$ t(\vec{w}, \vec{u}, \vec{v}) = \oint_{\Gamma} \hat F \cdot \llbracket \vec{u} \rrbracket \ \textrm{d}A - \int_{\Omega} (\vec{w} \otimes \vec{v}) : \nabla \vec{u} \ \textrm{d}V $$
For the flux $\hat F$ one might e.g. use an upwind-flux. The upwind-direction can be identified based on the velocity $\{ \vec{u} \}$.

**Treatment of Nonlinear Terms:** We have to solve: find $(\vec{u}, p) \in \mathbb{P}_k(\mathfrak{k}_h)^D \times \mathbb{P}_k(\mathfrak{k}_h)$ so that
>$$ \begin{align} t(\vec{u}, \vec{u}, \vec{v}) + a(\vec{u}, \vec{v}) + b(p, \vec{v}) &= \int_{\Omega} \vec{f} \cdot \vec{v} \ \textrm{d}V \quad \forall \vec{v} \in \mathbb{P}_{k}(\mathfrak{K}_h)^D \\ -b(\tau, \vec{u}) + \textrm{Re} \ s(p,\tau) &= 0 \quad \forall \tau \in \mathbb{P}_{k}(\mathfrak{K}_h) \end{align}$$
Obviously, this problem is nonlinear. Therefore it has to be solved iteratively. We search for a sequence
>$$ (\vec{u}^1, p^1),(\vec{u}^2, p^2) \dots \rightarrow (\vec{u}, p) $$
which converges to the solution $(\vec{u}, p)$ of the above problem. In the $n$-th iteration $(n = 1, 2, . . . )$ we
solve
>$$ \begin{align} t(\vec{u}^{n-1}, \vec{u}^\ast, \vec{v}) + a(\vec{u}^\ast, \vec{v}) + b(p^\ast, \vec{v}) &= \int_{\Omega} \vec{f} \cdot \vec{v} \ \textrm{d}V \quad \forall \vec{v} \in \mathbb{P}_{k}(\mathfrak{K}_h)^D \\ -b(\tau, \vec{u}^\ast) + \textrm{Re} \ s(p^\ast,\tau) &= 0 \quad \forall \tau \in \mathbb{P}_{k}(\mathfrak{K}_h) \end{align}$$
and define
>$$ \begin{align} \vec{u}^n = (\alpha_u - 1) \vec{u}^{n-1} + \alpha_u \vec{u}^\ast \\ p^n = (\alpha_p - 1) p^{n-1} + \alpha_p p^\ast\end{align}$$
The under-relaxation factors $\alpha_u$ and $\alpha_p$ , $0 < \alpha_u , \alpha_p \leq 1$ can be tuned to improve the convergence behavior of the method. As initial values one might just use $\vec{u}^0 = 0, p^0 = 0$. Most of the time, one might just use $\alpha_u = \alpha_p = 1$. In this case, the iterative procedure can be interpreted as a Picard-iteration or a fixpointiteration. Alternatively, it may be called Oseen iteration.